In [ ]:
# Demo 6: Test claim extraction with cleaned/chunked transcript (chunk batching)

# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api

In [ ]:
# Libraries
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from transformers import AutoTokenizer

# Integrate youtube_api.py code:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# Load tokenizer with left padding for Qwen
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side ='left')



In [ ]:

# Integrate youtube_api.py code:

load_dotenv()

api_key= os.getenv("API_KEY")

yta = YouTubeTranscriptApi()

"""
def get_transcript(video_id):
    try:
        transcript = yta.fetch(video_id)
        transcript_text = " ".join(entry.text for entry in transcript)
        return transcript_text
    except Exception:
        return None
"""

youtube = build('youtube', 'v3', developerKey=api_key)

request = youtube.videos().list(
    part='localizations,statistics,topicDetails',
    chart='mostPopular',
    regionCode='us',
    videoCategoryId='20',
    maxResults='50'
)

response = request.execute()

videos_data = []

for item in response["items"]:
    video_id = item.get("id")

    videos_data.append({"Video ID": video_id})

df = pd.DataFrame(videos_data)
# df["Transcript"] = df["Video ID"].apply(get_transcript)


In [ ]:
# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    tokenizer = tokenizer,
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200 # increase to prevent incomplete output
)

In [ ]:
"""
# Test retrieve youtube video transcript (for 1 video)

video_id = "TT4HTDYhiOU" # sample video ID
yta = YouTubeTranscriptApi()
transcript = yta.fetch(video_id)
print(f"Transcript:\n{transcript}")
"""


Output Format:

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='my little brother recently got Minecraft', start=0.04, duration=3.639), ...

Format Analysis:

- FetchedTranscript object (holds snippets list)
  - snippets list (holds FetchedTranscriptSnippet objects (each object is a snippet))
    - FetchedTranscriptSnippet object (holds 3 attributes: text, start, duration)

In [ ]:
# Test clean raw transcript

def clean_transcript(transcript):
  cleaned = [] # list for cleaned snippets

  # Loop to get text (clean) & get start time
  for snippet in transcript.snippets:
    text = snippet.text # get text from snippet
    start_time = snippet.start # get start time from snippet

    while "[" in text:
      start = text.index("[")
      end = text.index("]") + 1
      caption = text[start:end]
      text = text.replace(caption, "")

    cleaned.append({"text": text, "start": start_time}) # add snippet's text and start time to cleaned list

  # Test print cleaned list
  # print(f"Transcript Text:\n{cleaned}")

  return cleaned

Get cleaned transcript text & start time for each snippet

Transcript Text:

[{'text': 'So guys, Minecraft 26.0 is finally out,', 'start': 0.08}, {'text': 'and this update brings more than 50', 'start': 3.2},...]

In [ ]:
# Test chunking cleaned transcript

def chunk_transcript(cleaned):
  chunk_size = 300 # chunk size based on time (sec)
  chunks = [] # list for chunks
  chunk_snippets = [] # list to build each chunk
  curr_chunk_time = 0

  # Loop through all snippets in cleaned[]
  for snippet in cleaned:
      text = snippet["text"] # get snippet text
      chunk_snippets.append(text) # add text to chunk_snippets list

      # Check if reach chunk_size (approx.)
      if snippet["start"] - curr_chunk_time >= chunk_size:
        # print(f"Chunk time: {snippet["start"] - curr_chunk_time} sec") # Test print chunk start time
        chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
        chunks.append(chunk) # add chunk to list of chunks

        chunk_snippets = [] # reset chunk_snippets list
        curr_chunk_time = snippet["start"] # reset time to current start time

  # If leftover snippets, create and add last chunk
  if chunk_snippets:
    chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
    chunks.append(chunk) # add chunk to list of chunks

  # Test print chunks list
  # print(f"Transcript Text:\n{chunks}")

  return chunks


Test chunk time:(e.g. chunk size = 60 sec)

Chunk time: 60.64 sec

Chunk time: 61.28 sec

Chunk time: 61.599000000000004 sec

Chunk time: 61.601 sec

Chunk time: <60 sec

In [ ]:
# Test cleaned/chunked transcript in claim extraction step (chunk batching)

# Function to extract claims
def claim_extraction(chunks):
  message_list = []
  # Create message more each chunk
  for chunk in chunks:
    messages = [
      {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
      # {"role": "user", "content": f"Extract all factual claims as bullet points.\n\nText: {chunk}"}
      {"role": "user", "content": f"Extract one concise high-level claim that summarizes the main point.\n\nText: {chunk}"}
    ]
    message_list.append(messages) # add message to list

  results = []
  # Send messages to Qwen model (process in batches)
  for output in pipe(message_list, batch_size=5, max_length=None):
    # Get claim from Qwen response
    result = output[0]["generated_text"][-1]["content"]
    results.append(result) # add qwen output to list
  return results


In [ ]:
# Test extract claim from multiple videos

# json library
import json

# video sample list test time filter: 30 minutes
# video_ids = ["NgP8_TI0qD0", "vcRoRl-CQys", "YV-576jC1BU",] # list of sample video ids (25min, 42min, 31min)

# 50 video ids from df into a list test:
video_ids = df["Video ID"].tolist()[:10] # test first 10
# video_ids = df["Video ID"].tolist() # test all 50

video_claims = []
# Loop through each video id in list
for video_id in video_ids:
  yta = YouTubeTranscriptApi()
  try:
    transcript = yta.fetch(video_id) # get transcript

    # Filter videos based on time (keep only <=30 minutes)
    video_time = transcript.snippets[-1].start + transcript.snippets[-1].duration
    if video_time <= 1800:

      chunks = chunk_transcript(clean_transcript(transcript)) # clean/chunk transcript

      # Extract claims from chunks list
      claims_list = claim_extraction(chunks)

      video_claims.append({"video_id": video_id, "claims": claims_list})
      """
      # Print claims
      for claims in claims_list:
        print(f"Claims Extracted:\n{claims}\n")
      """
    """
    else:
      print(f"Video ID: {video_id} is too long")
    """

  except Exception: # error for no transcript
    print(f"No transcript for video ID: {video_id}")

with open('video_claims.json', 'w') as file:
  json.dump(video_claims, file, indent=4)



video sample list test time filter for <=30 minutes: (25min, 42min, 31min)

25 min video:

Claims Extracted:
- Started playing Vintage Story in February.
- Rarely compares Vintage Story to Minecraft due to feeling it surpasses Minecraft.
- Analogizes Vintage Story to Windows vs. Mac OS to highlight differences.
- Intends to present Vintage Story independently of Minecraft without causing arguments online.
- Discusses the goals of Minecraft (creating personal experiences) and Vintage Story (better creation tools).
- Describes Mojang's efforts to improve Minecraft's development and customization capabilities.
- Explains changes from version 1.20.4 to 1.20.5 in terms of creating servers.

Claims Extracted:
- Vintage Story is considered better than Minecraft.
- Survival mode in Minecraft is criticized due to the Hunger Bar mechanic, which allows players to sustain themselves indefinitely without risk of starvation.
- Players suggest that Vintage Story provides a more realistic survival experience by focusing on practical food sources and minimalistic gameplay mechanics.

Claims Extracted:
- Vintage Story's hunger system is described as being much more important than in Minecraft.
- The player must eat a variety of foods to gain health points and avoid boredom.
- Cooking in Vintage Story requires using ingredients such as meat, vegetables, and bones.
- The game offers a vast array of meals, including stews, soups, and porridge.
- Players can build greenhouses to extend crop growth during winter months.
- Chickens raised by players can be used to lay eggs and provide milk for crafting purposes.
- The game encourages interaction with the environment through farming and animal management.
- Customization options allow players to adjust aspects of their gameplay experience within the game.

Claims Extracted:
- Hard: The game has preset world generation options.
- Exploration vs Wilderness Survival: "Think of exploration as the easy mode and wilderness survival as the hard mode."
- Homo Sapien: A preset that feels similar to early Minecraft experiences.
- Customization: Can significantly alter the world and gameplay experience.
- Modding API: The Vintage Story modding platform provides tools for creating fan content.
- Modding Tools: Fabric and Quilt are alternative solutions to Forge for modding Minecraft.
- Community Contribution: Vintage Story has contributed significantly to the development of Minecraft through modding APIs.
- Performance Issues: Minecraft faces significant performance issues, especially on low-end hardware, which Mojang refuses to address.
- Gaming Experience: Players feel less free and creative compared to earlier versions of Minecraft.
- Marketplace Concerns: The lack of a proper modding API for Minecraft has led to community-developed solutions becoming dominant.

Claims Extracted:
- Vintage Story features an official modding API.
- Mods are downloaded automatically upon joining a server with mods.
- Users can make mods without extensive coding knowledge due to its simplicity compared to Minecraft.
- Open-source modeling and animation tools are available for users to build upon.
- There is no marketplaces or paid DLC in Vintage Story.
- The game is supported by a single optional supporter add-on offering a Discord role and golden name.
- The game costs €21 (approximately $24 USD) and supports indie developers financially.
- Vintage Story fosters a supportive and passionate community, unlike Minecraft's toxic atmosphere.
- The Discord serves as a hub for community interaction and resource sharing.
- The development team at IndieStudios is known for respecting user wallets and opinions.

Claims Extracted:
- The player is looking forward to seeing how the game develops.
- There is an external link provided in the description for purchasing the game.
- Joining the Discord might allow the player to interact with the community more frequently.
- Members of the Discord pay $3 per month to support the creator.

42 & 31 min video:

Video ID: vcRoRl-CQys is too long

Video ID: YV-576jC1BU is too long